In [9]:
import pandas as pd
import glob
import os
from pathlib import Path

In [10]:
def get_batch_number(filename):
    """Extract batch number from filename like filtered_events_batch_00000.csv"""
    basename = os.path.basename(filename)
    batch_str = basename.replace('filtered_events_batch_', '').replace('.csv', '')
    return int(batch_str)

In [12]:
def count_events_in_time_window(folder_path, start_time, end_time):
    """
    Count events between start_time and end_time from multiple CSV files.
    Stops processing when timestamps exceed end_time since files are in temporal order.
    
    Args:
        folder_path: Path to folder containing filtered_events_batch_*.csv files
        start_time: Starting timestamp (inclusive)
        end_time: Ending timestamp (inclusive)
    
    Returns:
        total_events: Number of events in the time window
        processed_files: List of files that were processed
    """
    
    # Get all CSV files matching the pattern
    file_pattern = os.path.join(folder_path, "filtered_events_batch_*.csv")
    all_files = glob.glob(file_pattern)
    
    # Sort files by batch number
    sorted_files = sorted(all_files, key=get_batch_number)
    
    total_events = 0
    processed_files = []
    stopped_early = False
    
    print(f"Searching for events between {start_time} and {end_time}")
    print("-" * 50)
    
    for file_path in sorted_files:
        batch_num = get_batch_number(file_path)
        print(f"Processing batch {batch_num:05d}...")
        
        # Read only timestamp column first to check time range
        try:
            # Read timestamps to check if file is relevant
            timestamps = pd.read_csv(file_path, usecols=['timestamp'])
            
            # Check if any timestamp in this file is within or before our window
            min_ts = timestamps['timestamp'].min()
            max_ts = timestamps['timestamp'].max()
            
            print(f"  Time range: {min_ts:.6f} - {max_ts:.6f}")
            
            # If max timestamp in file is less than start_time, skip (too early)
            if max_ts < start_time:
                print(f"  Skipping - all events before start time")
                processed_files.append(file_path)
                continue
            
            # If min timestamp is greater than end_time, we can stop
            '''if min_ts > end_time:
                print(f"  Stopping - all remaining events after end time")
                stopped_early = True
                processed_files.append(file_path)
                break'''
            
            # File has events in our window - read full file and filter
            df = pd.read_csv(file_path)
            
            # Filter events in time window
            mask = (df['timestamp'] >= start_time) & (df['timestamp'] <= end_time)
            events_in_file = mask.sum()
            
            if events_in_file > 0:
                print(f"  Found {events_in_file} events in window")
                total_events += events_in_file
            else:
                print(f"  No events in window")
            
            processed_files.append(file_path)
            
        except Exception as e:
            print(f"  Error processing file: {e}")
            continue
    
    print("-" * 50)
    print(f"Total events found: {total_events}")
    print(f"Files processed: {len(processed_files)}")
    if stopped_early:
        print("Stopped early (reached time limit)")
    
    return total_events, processed_files

In [13]:
# Your parameters
folder_path = r"C:\Arjun\Thesis\data\20200421_170039-sunset1\filtered chunks\subsampled"
start_time = 1587452585.688726
end_time = 1587452595.688726

# Count events
event_count, files = count_events_in_time_window(folder_path, start_time, end_time)

print(f"\nFinal result: {event_count} events between {start_time} and {end_time}")

Searching for events between 1587452585.688726 and 1587452595.688726
--------------------------------------------------
Processing batch 00000...
  Time range: 1587452583.688726 - 1587452589.770837
  Found 37197 events in window
Processing batch 00001...
  Time range: 1587452589.774911 - 1587452592.455371
  Found 44722 events in window
Processing batch 00002...
  Time range: 1587452592.477207 - 1587452595.783045
  Found 28786 events in window
Processing batch 00003...
  Time range: 1587452595.787656 - 1587452598.794834
  No events in window
Processing batch 00004...
  Time range: 1587452598.798971 - 1587452602.056395
  No events in window
Processing batch 00005...
  Time range: 1587452602.064426 - 1587452604.121848
  No events in window
Processing batch 00006...
  Time range: 1587452604.127446 - 1587452605.498458
  No events in window
Processing batch 00007...
  Time range: 1587452605.501631 - 1587452606.805057
  No events in window
Processing batch 00008...
  Time range: 1587452606.81

In [3]:
s_time = 1587540266.671854
end_time = 1587540271.671854
print(end_time - s_time)

5.0


In [4]:
# Filter events between the two timestamps
filtered_df = df[(df['timestamp'] >= s_time) & (df['timestamp'] <= end_time)]

# Count the number of events
event_count = len(filtered_df)
print(f"Number of events between {s_time} and {end_time}: {event_count}")

Number of events between 1587540266.671854 and 1587540271.671854: 232265
